# DeepDTA on KIBA (CNN, as in the base paper)

Run the cells **in order, top to bottom**. Before you start: **Runtime > Change runtime type > GPU**.

What this notebook does:
1. Loads the KIBA data from the DeepDTA GitHub repo and uses the paper's train and test folds.
2. Trains the paper's CNN model for 100 epochs (SMILES length 100, protein length 1000).
3. Measures CI, MSE, r2m and AUPR on the test fold.
4. Saves test results for the accuracy graph and 5 example pairs for the website.
5. Converts the trained model to ONNX, checks it against the original, and downloads one ZIP file.

Training is the long part (roughly 1.5 to 3 hours on a free GPU). Keep this browser tab open.

## 1. Setup

In [1]:
import os, json, pickle, shutil, glob
from collections import OrderedDict

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, average_precision_score

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPU found:", gpus if gpus else "NO GPU. Go to Runtime > Change runtime type > GPU, then run this cell again.")

TensorFlow: 2.21.0
GPU found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Settings (KIBA values from the paper)

In [2]:
MAX_SMILES_LENGTH = 100       # Section 2.3: KIBA
MAX_PROTEIN_LENGTH = 1000     # Section 2.3: KIBA
EMBEDDING_DIM = 128           # Section 2.3
NUM_FILTERS = 32              # 32, 64, 96 filters (Section 3.3)
SMILES_KERNEL = 4             # inside the paper's range (Table 2)
PROTEIN_KERNEL = 8            # inside the paper's range (Table 2)
DENSE_UNITS = (1024, 1024, 512)
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
BATCH_SIZE = 256
EPOCHS = 100
KIBA_THRESHOLD = 12.1         # Section 3.4: KIBA score >= 12.1 counts as binding (used for AUPR)
print("Settings ready")

Settings ready


## 3. Get the data from GitHub

In [3]:
ZIP = "/content/deepdta.zip"
!wget -q https://github.com/hkmztrk/DeepDTA/archive/refs/heads/master.zip -O /content/deepdta.zip

if not os.path.exists(ZIP) or os.path.getsize(ZIP) < 1_000_000:
    from google.colab import files
    print("Download from GitHub failed. Upload DeepDTA-master.zip from your computer:")
    up = files.upload()
    ZIP = "/content/" + list(up.keys())[0]

!unzip -q -o "{ZIP}" -d /content/deepdta_repo
KIBA_DIR = glob.glob("/content/deepdta_repo/**/data/kiba", recursive=True)[0]
print("Using:", KIBA_DIR)
print(os.listdir(KIBA_DIR))

Using: /content/deepdta_repo/DeepDTA-master/data/kiba
['kiba_drug_sim.txt', 'kiba_target_sim.txt', 'ligands_iso.txt', 'Y', 'proteins.txt', 'kiba_binding_affinity_v2.txt', 'folds', 'ligands_can.txt']


## 4. Load drugs, proteins and scores

In [4]:
with open(os.path.join(KIBA_DIR, "ligands_can.txt")) as f:
    ligands = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(KIBA_DIR, "proteins.txt")) as f:
    proteins = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(KIBA_DIR, "Y"), "rb") as f:
    Y = pickle.load(f, encoding="latin1")

drug_ids = list(ligands.keys())
protein_ids = list(proteins.keys())
drugs = list(ligands.values())       # SMILES strings
targets = list(proteins.values())    # amino-acid sequences
affinity = np.array(Y, dtype=np.float64)   # KIBA scores. NaN = never measured. No transformation for KIBA.

print("Drugs:", len(drugs), "| Proteins:", len(targets))
print("Y shape:", affinity.shape)
assert affinity.shape == (len(drugs), len(targets)), "Y does not match the drug and protein lists"

drug_idx, target_idx = np.where(~np.isnan(affinity))
print("Known drug-target pairs:", len(drug_idx), "(paper: 118,254)")
print("KIBA score range:", round(np.nanmin(affinity), 3), "to", round(np.nanmax(affinity), 3))
assert len(drug_idx) == 118254, "Pair count differs from the paper's Table 1"

Drugs: 2111 | Proteins: 229
Y shape: (2111, 229)
Known drug-target pairs: 118254 (paper: 118,254)
KIBA score range: 0.0 to 17.2


## 5. Paper's train and test folds, and the number encoding

In [5]:
with open(os.path.join(KIBA_DIR, "folds", "train_fold_setting1.txt")) as f:
    train_folds = json.load(f)
with open(os.path.join(KIBA_DIR, "folds", "test_fold_setting1.txt")) as f:
    test_fold = json.load(f)

train_positions = np.array([p for fold in train_folds for p in fold])
test_positions = np.array(test_fold)

train_drugs, train_targets = drug_idx[train_positions], target_idx[train_positions]
test_drugs, test_targets = drug_idx[test_positions], target_idx[test_positions]
y_train = affinity[train_drugs, train_targets]
y_test = affinity[test_drugs, test_targets]
print("Training pairs:", len(y_train), "| Test pairs:", len(y_test))
assert len(set(train_positions) & set(test_positions)) == 0, "Train and test overlap"

smiles_chars = sorted(set("".join(drugs)))
protein_chars = sorted(set("".join(targets)))
smiles_vocab = {ch: i + 1 for i, ch in enumerate(smiles_chars)}
protein_vocab = {ch: i + 1 for i, ch in enumerate(protein_chars)}

print("SMILES symbols:", len(smiles_vocab), "->", "".join(smiles_chars))
print("Contains lowercase aromatic letters (c, n, o, s):", [c for c in "cnos" if c in smiles_vocab])
print("Contains '[', '@', '/' or backslash:", [c for c in "[@/\\" if c in smiles_vocab])
print("Protein symbols:", len(protein_vocab), "->", "".join(protein_chars))

smiles_style = {
    "lowercase_aromatic": any(c in smiles_vocab for c in "cnosp"),
    "brackets": "[" in smiles_vocab,
    "stereo_chars": any(c in smiles_vocab for c in "@/\\"),
    "plus_or_minus": any(c in smiles_vocab for c in "+-"),
}
print("SMILES style found in this dataset:", smiles_style)

def encode(sequence, vocab, max_len):
    encoded = np.zeros(max_len, dtype=np.int32)   # zeros = padding
    for i, ch in enumerate(sequence[:max_len]):
        encoded[i] = vocab.get(ch, 0)
    return encoded

encoded_drugs = np.array([encode(s, smiles_vocab, MAX_SMILES_LENGTH) for s in drugs])
encoded_targets = np.array([encode(s, protein_vocab, MAX_PROTEIN_LENGTH) for s in targets])

X_drug_train, X_target_train = encoded_drugs[train_drugs], encoded_targets[train_targets]
X_drug_test, X_target_test = encoded_drugs[test_drugs], encoded_targets[test_targets]
print("Input shapes:", X_drug_train.shape, X_target_train.shape)

Training pairs: 98545 | Test pairs: 19709
SMILES symbols: 29 -> #()+-.123456789=BCFHINOPS[]lr
Contains lowercase aromatic letters (c, n, o, s): []
Contains '[', '@', '/' or backslash: ['[']
Protein symbols: 20 -> ACDEFGHIKLMNPQRSTVWY
SMILES style found in this dataset: {'lowercase_aromatic': False, 'brackets': True, 'stereo_chars': False, 'plus_or_minus': True}
Input shapes: (98545, 100) (98545, 1000)


## 6. Optional: keep checkpoints in Google Drive
If Drive connects, training progress survives a disconnect. **If it shows an error, ignore it and carry on**: checkpoints then stay in Colab's temporary storage.

In [6]:
WORK_DIR = "/content/kiba_work"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/DeepDTA_project/kiba_work"
    print("Drive connected.")
except Exception as e:
    print("Drive not connected, using temporary storage instead:", type(e).__name__)
os.makedirs(WORK_DIR, exist_ok=True)
print("Work folder:", WORK_DIR)

Mounted at /content/drive
Drive connected.
Work folder: /content/drive/MyDrive/DeepDTA_project/kiba_work


## 7. Build the model (two CNN blocks, as in the paper)

In [7]:
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model

def cnn_branch(name, max_len, vocab_size, kernel_size):
    inp = Input(shape=(max_len,), name=name + "_input")
    x = Embedding(input_dim=vocab_size + 1, output_dim=EMBEDDING_DIM, name=name + "_embedding")(inp)
    for i, mult in enumerate((1, 2, 3), start=1):
        x = Conv1D(NUM_FILTERS * mult, kernel_size, activation="relu", name=f"{name}_conv{i}")(x)
    x = GlobalMaxPooling1D(name=name + "_pool")(x)
    return inp, x

drug_input, drug_features = cnn_branch("drug", MAX_SMILES_LENGTH, len(smiles_vocab), SMILES_KERNEL)
protein_input, protein_features = cnn_branch("protein", MAX_PROTEIN_LENGTH, len(protein_vocab), PROTEIN_KERNEL)

x = Concatenate(name="merge")([drug_features, protein_features])
x = Dense(DENSE_UNITS[0], activation="relu", name="fc1")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[1], activation="relu", name="fc2")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[2], activation="relu", name="fc3")(x)
output = Dense(1, activation="linear", name="score")(x)

model = Model(inputs=[drug_input, protein_input], outputs=output, name="DeepDTA_KIBA")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse")
model.summary()

Model: "DeepDTA_KIBA"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ drug_input          │ (None, 100)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_input       │ (None, 1000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drug_embedding      │ (None, 100, 128)  │      3,840 │ drug_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_embedding   │ (None, 1000, 128) │      2,688 │ protein_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drug_conv1 (Conv1D) │ (None, 97, 32)    │     16,416 │ drug_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_conv1       │ (None, 993, 32)   │     32,800 │ protein_embeddin… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drug_conv2 (Conv1D) │ (None, 94, 64)    │      8,256 │ drug_conv1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_conv2       │ (None, 986, 64)   │     16,448 │ protein_conv1[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drug_conv3 (Conv1D) │ (None, 91, 96)    │     24,672 │ drug_conv2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_conv3       │ (None, 979, 96)   │     49,248 │ protein_conv2[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drug_pool           │ (None, 96)        │          0 │ drug_conv3[0][0]  │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ protein_pool        │ (None, 96)        │          0 │ protein_conv3[0]… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ merge (Concatenate) │ (None, 192)       │          0 │ drug_pool[0][0],  │
│                     │                   │            │ protein_pool[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc1 (Dense)         │ (None, 1024)      │    197,632 │ merge[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 1024)      │          0 │ fc1[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc2 (Dense)         │ (None, 1024)      │  1,049,600 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 1024)      │          0 │ fc2[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc3 (Dense)         │ (None, 512)       │    524,800 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score (Dense)       │ (None, 1)         │        513 │ fc3[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,926,913 (7.35 MB)

 Trainable params: 1,926,913 (7.35 MB)

 Non-trainable params: 0 (0.00 B)

## 8. Train for 100 epochs
The model is saved after every epoch. **If the session stops partway** and Colab is still connected, run cells 1 to 7 again and then this cell: it resumes from the last saved epoch.

In [8]:
CKPT = os.path.join(WORK_DIR, "kiba_last.keras")
LOG = os.path.join(WORK_DIR, "kiba_training_log.csv")

start_epoch = 0
if os.path.exists(CKPT) and os.path.exists(LOG):
    start_epoch = len(pd.read_csv(LOG))
    model = tf.keras.models.load_model(CKPT)
    print("Resuming from epoch", start_epoch)

if start_epoch >= EPOCHS:
    print("Training already finished.")
else:
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(CKPT, save_best_only=False),
        tf.keras.callbacks.CSVLogger(LOG, append=True),
    ]
    model.fit(
        [X_drug_train, X_target_train], y_train,
        epochs=EPOCHS, initial_epoch=start_epoch,
        batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2,
    )
    print("Training finished.")

Epoch 1/100
385/385 - 80s - 209ms/step - loss: 1.9713
Epoch 2/100
385/385 - 47s - 123ms/step - loss: 0.6134
Epoch 3/100
385/385 - 47s - 122ms/step - loss: 0.5921
Epoch 4/100
385/385 - 47s - 122ms/step - loss: 0.5323
Epoch 5/100
385/385 - 47s - 122ms/step - loss: 0.5062
Epoch 6/100
385/385 - 47s - 122ms/step - loss: 0.5075
Epoch 7/100
385/385 - 47s - 122ms/step - loss: 0.4680
Epoch 8/100
385/385 - 47s - 122ms/step - loss: 0.4490
Epoch 9/100
385/385 - 47s - 122ms/step - loss: 0.4193
Epoch 10/100
385/385 - 47s - 122ms/step - loss: 0.3941
Epoch 11/100
385/385 - 47s - 122ms/step - loss: 0.3701
Epoch 12/100
385/385 - 47s - 122ms/step - loss: 0.3612
Epoch 13/100
385/385 - 47s - 122ms/step - loss: 0.3504
Epoch 14/100
385/385 - 47s - 122ms/step - loss: 0.3490
Epoch 15/100
385/385 - 47s - 122ms/step - loss: 0.3280
Epoch 16/100
385/385 - 47s - 122ms/step - loss: 0.3195
Epoch 17/100
385/385 - 47s - 122ms/step - loss: 0.3084
Epoch 18/100
385/385 - 47s - 122ms/step - loss: 0.3006
Epoch 19/100
385/38

## 9. Test results (the test fold was never used for training)

In [9]:
def concordance_index(y_true, y_pred):
    # Paper Eq. (7)-(8): share of pairs ranked in the right order; ties count 0.5
    total, pairs = 0.0, 0
    for i in range(len(y_true)):
        larger = y_true[i] > y_true
        diff = y_pred[i] - y_pred[larger]
        total += np.sum(diff > 0) + 0.5 * np.sum(diff == 0)
        pairs += np.sum(larger)
    return total / pairs

def r2m(y_obs, y_pred):
    # r2m as written in the DeepDTA repo's emetrics.py (observed values regressed on predictions)
    y_obs, y_pred = np.asarray(y_obs), np.asarray(y_pred)
    r2 = (np.sum((y_pred - y_pred.mean()) * (y_obs - y_obs.mean())) ** 2) / (
        np.sum((y_obs - y_obs.mean()) ** 2) * np.sum((y_pred - y_pred.mean()) ** 2))
    k = np.sum(y_obs * y_pred) / np.sum(y_pred * y_pred)
    r02 = 1 - np.sum((y_obs - k * y_pred) ** 2) / np.sum((y_obs - y_obs.mean()) ** 2)
    return r2 * (1 - np.sqrt(abs(r2 * r2 - r02 * r02)))

y_pred = model.predict([X_drug_test, X_target_test], batch_size=512, verbose=0).ravel()

ci = concordance_index(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2m_value = r2m(y_test, y_pred)
aupr = average_precision_score((y_test >= KIBA_THRESHOLD).astype(int), y_pred)

print("========== KIBA TEST RESULTS ==========")
print("CI  :", round(ci, 4),  " (paper: 0.863)")
print("MSE :", round(mse, 4), " (paper: 0.194)")
print("r2m :", round(r2m_value, 4), " (paper: 0.673)")
print("AUPR:", round(aupr, 4), " (paper: 0.788)")

========== KIBA TEST RESULTS ==========
CI  : 0.8713  (paper: 0.863)
MSE : 0.1703  (paper: 0.194)
r2m : 0.7052  (paper: 0.673)
AUPR: 0.8104  (paper: 0.788)


## 10. Save the graph data and 5 example pairs

In [10]:
OUT = os.path.join(WORK_DIR, "package", "kiba")
os.makedirs(OUT, exist_ok=True)

# metrics
metrics = {"dataset": "KIBA", "epochs": EPOCHS, "n_test": int(len(y_test)),
           "CI": round(float(ci), 4), "MSE": round(float(mse), 4),
           "r2m": round(float(r2m_value), 4), "AUPR": round(float(aupr), 4),
           "score_percentiles": {str(p): round(float(np.percentile(y_test, p)), 3) for p in (5, 25, 50, 75, 95)},
           "share_at_or_above_cutoff": round(float((y_test >= KIBA_THRESHOLD).mean()), 3)}
json.dump(metrics, open(os.path.join(OUT, "metrics.json"), "w"))

# 3,000 randomly chosen test pairs for the website graph: [predicted, measured]
rng = np.random.default_rng(0)
pick = rng.choice(len(y_test), size=min(3000, len(y_test)), replace=False)
points = [[round(float(y_pred[i]), 3), round(float(y_test[i]), 3)] for i in pick]
json.dump(points, open(os.path.join(OUT, "test_points.json"), "w"))

# 5 example pairs from the TEST fold, picked by measured value (not by how well the model did)
order = np.argsort(y_test)
examples = []
for q in (0.10, 0.30, 0.50, 0.70, 0.90):
    i = int(order[int(q * (len(order) - 1))])
    d, t = int(test_drugs[i]), int(test_targets[i])
    examples.append({
        "drug_id": drug_ids[d], "protein_id": protein_ids[t],
        "smiles": drugs[d], "sequence": targets[t],
        "measured": round(float(y_test[i]), 3),
    })
json.dump(examples, open(os.path.join(OUT, "examples.json"), "w"))

print(metrics)
for e in examples:
    print(e["drug_id"], "x", e["protein_id"], "| measured:", e["measured"], "| SMILES length:", len(e["smiles"]), "| protein length:", len(e["sequence"]))

{'dataset': 'KIBA', 'epochs': 100, 'n_test': 19709, 'CI': 0.8713, 'MSE': 0.1703, 'r2m': 0.7052, 'AUPR': 0.8104, 'score_percentiles': {'5': 10.702, '25': 11.2, '50': 11.5, '75': 11.9, '95': 13.437}, 'share_at_or_above_cutoff': 0.207}
CHEMBL1989474 x Q5VT25 | measured: 11.1 | SMILES length: 39 | protein length: 1732
CHEMBL2000393 x P49137 | measured: 11.2 | SMILES length: 60 | protein length: 400
CHEMBL2002992 x Q9UEE5 | measured: 11.5 | SMILES length: 38 | protein length: 414
CHEMBL1979252 x P78368 | measured: 11.9 | SMILES length: 52 | protein length: 415
CHEMBL462120 x P07333 | measured: 12.8 | SMILES length: 42 | protein length: 972


## 11. Convert to ONNX and check it against the trained model

In [12]:
from google.colab import drive
drive.mount('/content/drive')
model.save('/content/drive/MyDrive/deepdta_kiba_backup.keras')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip install onnx onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 122.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 102.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [onnx]


In [14]:
import onnx, onnxruntime as ort
from onnx import helper, TensorProto, numpy_helper

model.save(os.path.join(OUT, "deepdta_kiba.keras"))

config = {
    "dataset": "KIBA", "unit": "KIBA score", "binds_cutoff": KIBA_THRESHOLD,
    "max_smiles_length": MAX_SMILES_LENGTH, "max_protein_length": MAX_PROTEIN_LENGTH,
    "smiles_vocab": smiles_vocab, "protein_vocab": protein_vocab,
    "smiles_style": smiles_style,
}
json.dump(config, open(os.path.join(OUT, "config.json"), "w"))

nodes, inits = [], []
def init(name, arr):
    inits.append(numpy_helper.from_array(np.asarray(arr, dtype=np.float32), name))
    return name

def branch(prefix):
    in_name = prefix + "_input"
    emb_w = model.get_layer(prefix + "_embedding").get_weights()[0]
    nodes.append(helper.make_node("Gather", [init(prefix + "_emb", emb_w), in_name], [prefix + "_e"], axis=0))
    nodes.append(helper.make_node("Transpose", [prefix + "_e"], [prefix + "_t"], perm=[0, 2, 1]))
    x = prefix + "_t"
    for i in (1, 2, 3):
        w, b = model.get_layer(f"{prefix}_conv{i}").get_weights()
        w = np.transpose(w, (2, 1, 0))   # Keras (length, in, out) -> ONNX (out, in, length)
        nodes.append(helper.make_node("Conv", [x, init(f"{prefix}_cw{i}", w), init(f"{prefix}_cb{i}", b)], [f"{prefix}_c{i}"]))
        nodes.append(helper.make_node("Relu", [f"{prefix}_c{i}"], [f"{prefix}_r{i}"]))
        x = f"{prefix}_r{i}"
    nodes.append(helper.make_node("GlobalMaxPool", [x], [prefix + "_gp"]))
    nodes.append(helper.make_node("Flatten", [prefix + "_gp"], [prefix + "_f"], axis=1))
    return prefix + "_f"

d = branch("drug")
p = branch("protein")
nodes.append(helper.make_node("Concat", [d, p], ["cat"], axis=1))
x = "cat"
for ln in ("fc1", "fc2", "fc3", "score"):
    w, b = model.get_layer(ln).get_weights()
    out = "prediction" if ln == "score" else ln + "_o"
    nodes.append(helper.make_node("Gemm", [x, init(ln + "_w", w), init(ln + "_b", b)], [out]))
    if ln != "score":
        nodes.append(helper.make_node("Relu", [out], [ln + "_r"]))
        x = ln + "_r"

graph = helper.make_graph(
    nodes, "deepdta_kiba",
    [helper.make_tensor_value_info("drug_input", TensorProto.INT64, [None, MAX_SMILES_LENGTH]),
     helper.make_tensor_value_info("protein_input", TensorProto.INT64, [None, MAX_PROTEIN_LENGTH])],
    [helper.make_tensor_value_info("prediction", TensorProto.FLOAT, [None, 1])],
    initializer=inits,
)
onnx_model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 13)])
onnx_model.ir_version = 8
onnx.checker.check_model(onnx_model)
ONNX_FILE = os.path.join(OUT, "model.onnx")
onnx.save(onnx_model, ONNX_FILE)
print("ONNX file size:", round(os.path.getsize(ONNX_FILE) / 1e6, 2), "MB")

# Compare on 500 REAL test pairs
n = 500
keras_out = model.predict([X_drug_test[:n], X_target_test[:n]], verbose=0).ravel()
sess = ort.InferenceSession(ONNX_FILE, providers=["CPUExecutionProvider"])
onnx_out = sess.run(None, {"drug_input": X_drug_test[:n].astype(np.int64),
                           "protein_input": X_target_test[:n].astype(np.int64)})[0].ravel()
diff = float(np.max(np.abs(keras_out - onnx_out)))
print("Largest difference between Keras and ONNX:", diff)
print("Keras:", keras_out[:3], "| ONNX:", onnx_out[:3])
assert diff < 1e-4, "ONNX does not match Keras. Do NOT use this file. Send the output to Claude."
print("PASSED: ONNX gives the same answers as Keras.")

ONNX file size: 7.71 MB
Largest difference between Keras and ONNX: 5.7220458984375e-06
Keras: [14.605818 12.653412 11.484857] | ONNX: [14.60582  12.653416 11.484858]
PASSED: ONNX gives the same answers as Keras.


## 12. Download everything as one ZIP

In [15]:
from google.colab import files
zip_path = shutil.make_archive("/content/kiba_package", "zip", os.path.join(WORK_DIR, "package"))
print("ZIP size:", round(os.path.getsize(zip_path) / 1e6, 1), "MB")
files.download(zip_path)

ZIP size: 25.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>